# Photometric Analysis of NGC 1261 Globular Cluster Using HST FITS Images

In this python notebook, we will use FITS images of the globular cluster NGC 1261 taken
using the Hubble Space Telescope. We detect stars in the image, measure their magnitude
values using two filters, F275W and F336W, and generate a Colour Magnitude Diagram (CMD).

**Steps**
1. Open the F275W image, look at it, and find the stars
2. Measure the brightness of every star (aperture photometry) and convert to magnitude
3. Do the same for the F336W image
4. Match the stars between the two filters and plot the CMD

In [ ]:
!pip install -q photutils

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.stats import sigma_clipped_stats, SigmaClip

from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats, aperture_photometry

from scipy.spatial import cKDTree

In [ ]:
# Google Drive (Colab). If you are not on Colab, the data folder is just 'data/'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_folder = '/content/drive/MyDrive/Astropy_data/'
except ImportError:
    data_folder = 'data/'

file_275W = data_folder + 'hlsp_hugs_hst_wfc3-uvis_ngc1261_f275w_v1_stack-0859s (1).fits'
file_336W = data_folder + 'hlsp_hugs_hst_wfc3-uvis_ngc1261_f336w_v1_stack-0413s.fits'

## Part 1: F275W image

In [ ]:
fits_file_275W = fits.open(file_275W, memmap=True)
fits_file_275W.info()

In [ ]:
# extract data from the fits file
image_data_275W = fits_file_275W[0].data
print(image_data_275W)

In [ ]:
plt.figure()
plt.imshow(image_data_275W, origin = 'lower', cmap='Greys', norm=LogNorm())
plt.colorbar()
plt.show()

In [ ]:
# zoom to specific region
section1_275W = image_data_275W[4000:6000, 4000:6000]

plt.figure()
plt.imshow(section1_275W, origin = 'lower', cmap='Greys', norm=LogNorm())
plt.colorbar()
plt.show()

In [ ]:
header_275W = fits_file_275W[0].header
header_275W

### Which exposure time do we use?

The header has `EXPTIME = 7766 s`, but that is the **total** exposure time of all the images that went into the stack.
HUGS stacked images are an *average* of those exposures, scaled to the exposure time of the deepest image
used. That time is written in the file name (`..._stack-0859s.fits`), so this is the time we need in the magnitude formula.

The same applies to F336W (`stack-0413s`).

In [ ]:
filter_275W = header_275W['FILTER']

# read the number in 'stack-0859s' from the file name
t_stack_275W = float(re.search(r'stack-(\d+)s', file_275W).group(1))

print('Filter                          :', filter_275W)
print('Exposure time used (file name)  :', t_stack_275W)
print('EXPTIME in header (NOT used)    :', header_275W['EXPTIME'])

### Cluster centre on the image

The WCS (World Coordinate System) converts sky coordinates (RA, Dec) to pixel positions.
The zoomed section starts at pixel 4000, so the WCS has to be cut in the same way as the image,
otherwise the red cross lands in the wrong place.

In [ ]:
wcs_275W = WCS(header_275W)
wcs_section = wcs_275W[4000:6000, 4000:6000]   # same cut as the image section

center = SkyCoord(ra=header_275W['RA_TARG']*u.deg, dec=header_275W['DEC_TARG']*u.deg)
px, py = wcs_section.world_to_pixel(center)

plt.figure()
plt.subplot(projection=wcs_section)
plt.imshow(section1_275W, origin = 'lower', cmap='Greys', norm=LogNorm())
plt.plot(px, py, '+', color='red', markersize=10, markeredgewidth=2, label='NGC 1261 center')
plt.colorbar()
plt.legend()
plt.show()

## Detecting stars in F275W

First we find the background level using **sigma-clipped statistics** (stars are bright outliers, so they get removed).
The image has some empty pixels with the value `-25` (no data), these are ignored with a mask.

In [ ]:
no_data_275W = section1_275W <= -25      # empty pixels

mean_275W, median_275W, std_275W = sigma_clipped_stats(section1_275W, mask=no_data_275W, sigma=3.0)
print((mean_275W, median_275W, std_275W))

We do **not** mask the bright stars. They are real members of the cluster (red giants, horizontal branch stars)
and we want them in the CMD.

In [ ]:
daofind = DAOStarFinder(fwhm=3.0, threshold=5.0*std_275W)
sources_275W = daofind(section1_275W - median_275W, mask=no_data_275W)
print(sources_275W.colnames)
print(f"Stars detected: {len(sources_275W)}")

In [ ]:
# remove stars too close to the edge of the section (their background ring would fall outside the image)
margin = 15
ny, nx = section1_275W.shape

inside = ((sources_275W['x_centroid'] > margin) & (sources_275W['x_centroid'] < nx - 1 - margin) &
          (sources_275W['y_centroid'] > margin) & (sources_275W['y_centroid'] < ny - 1 - margin))
sources_275W = sources_275W[inside]
print(f"Stars after removing edge stars: {len(sources_275W)}")

## Setting Up Apertures

For each detected star we place:
- A **circular aperture** (radius = 6 pixels) to collect the star's light
- A **circular annulus** (ring from radius 10 to 15 pixels) around it to sample the local background

In [ ]:
positions_275W = np.transpose((sources_275W['x_centroid'], sources_275W['y_centroid']))
apertures_275W = CircularAperture(positions_275W, r=6.0)
annulus_aperture_275W = CircularAnnulus(positions_275W, r_in=10, r_out=15)

In [ ]:
# background per pixel from the ring, then multiply by the area of the aperture
sigma_clip = SigmaClip(sigma=3.0)
aperstats_275W = ApertureStats(section1_275W, annulus_aperture_275W, sigma_clip=sigma_clip)
bkg_mean_275W = aperstats_275W.mean

aperture_area_275W = apertures_275W.area_overlap(section1_275W)
total_bkg_275W = bkg_mean_275W * aperture_area_275W

In [ ]:
# perform aperture photometry
star_data_275W = aperture_photometry(section1_275W, apertures_275W)  # sum of the pixels of the stars

# our own x, y columns (same order as the stars) and the background
star_data_275W['x'] = positions_275W[:, 0]
star_data_275W['y'] = positions_275W[:, 1]
star_data_275W['total_bkg'] = total_bkg_275W
star_data_275W['net_flux'] = star_data_275W['aperture_sum'] - star_data_275W['total_bkg']

## Magnitudes

$$ \text{magnitude} = \text{zeropoint} - 2.5 \log_{10}\left(\frac{\text{net flux}}{t_{stack}}\right) $$

The zeropoints used in this notebook are taken from the official STScI WFC3 zeropoint table.

| Filter | ABMag Zeropoint |
|--------|----------------|
| F275W  | 24.158         |
| F336W  | 24.692         |

Source: [STScI WFC3 Zeropoints Notebook](https://spacetelescope.github.io/hst_notebooks/notebooks/WFC3/zeropoints/zeropoints.html)

In [ ]:
zeropoint_275W = 24.158

# remove stars with negative net flux before taking log
good_flux_275W = (star_data_275W['net_flux'] > 0) & np.isfinite(star_data_275W['net_flux'])
print(f"Stars removed due to negative flux: {(~good_flux_275W).sum()}")
print(f"Stars remaining: {good_flux_275W.sum()}")
star_data_275W = star_data_275W[good_flux_275W]

star_data_275W['magnitude'] = zeropoint_275W - 2.5 * np.log10(star_data_275W['net_flux'] / t_stack_275W)

for col in star_data_275W.colnames:
    star_data_275W[col].info.format = '%.8g'
star_data_275W[:10].pprint(max_width=-1)

## Part 2: F336W image

Exactly the same steps as F275W.

In [ ]:
fits_file_336W = fits.open(file_336W, memmap=True)
image_data_336W = fits_file_336W[0].data
header_336W = fits_file_336W[0].header

t_stack_336W = float(re.search(r'stack-(\d+)s', file_336W).group(1))
print('Filter                          :', header_336W['FILTER'])
print('Exposure time used (file name)  :', t_stack_336W)
print('EXPTIME in header (NOT used)    :', header_336W['EXPTIME'])

In [ ]:
plt.figure()
plt.imshow(image_data_336W, origin = 'lower', cmap='Greys', norm=LogNorm())
plt.colorbar()
plt.show()

## Overlaying F275W and F336W Images Using WCS

To confirm both filters cover the same sky region, we overlay the F275W (grey) and F336W (hot colormap) images using their respective WCS solutions.

In [ ]:
wcs_336W = WCS(header_336W)

fig = plt.figure()
ax = fig.add_subplot(111, projection=wcs_275W)  # base WCS is F275W

# Plot F275W as base image
ax.imshow(image_data_275W, origin='lower', cmap='Greys', norm=LogNorm(), alpha=0.9)

# Overlay F336W using its own WCS
ax.imshow(image_data_336W, origin='lower', cmap='hot', norm=LogNorm(), alpha=0.4,
          transform=ax.get_transform(wcs_336W))

ax.set_xlabel('Right Ascension')
ax.set_ylabel('Declination')
ax.set_title('NGC 1261 — F275W (grey) + F336W (hot) overlay')
plt.show()

In [ ]:
# same section as F275W
section1_336W = image_data_336W[4000:6000, 4000:6000]

plt.figure()
plt.imshow(section1_336W, origin = 'lower', cmap='Greys', norm=LogNorm())
plt.colorbar()
plt.show()

In [ ]:
no_data_336W = section1_336W <= -25

mean_336W, median_336W, std_336W = sigma_clipped_stats(section1_336W, mask=no_data_336W, sigma=3.0)
print((mean_336W, median_336W, std_336W))

daofind = DAOStarFinder(fwhm=3.0, threshold=5.0*std_336W)
sources_336W = daofind(section1_336W - median_336W, mask=no_data_336W)
print(f"Stars detected: {len(sources_336W)}")

# remove edge stars
inside = ((sources_336W['x_centroid'] > margin) & (sources_336W['x_centroid'] < nx - 1 - margin) &
          (sources_336W['y_centroid'] > margin) & (sources_336W['y_centroid'] < ny - 1 - margin))
sources_336W = sources_336W[inside]
print(f"Stars after removing edge stars: {len(sources_336W)}")

In [ ]:
positions_336W = np.transpose((sources_336W['x_centroid'], sources_336W['y_centroid']))
apertures_336W = CircularAperture(positions_336W, r=6.0)
annulus_aperture_336W = CircularAnnulus(positions_336W, r_in=10, r_out=15)

aperstats_336W = ApertureStats(section1_336W, annulus_aperture_336W, sigma_clip=sigma_clip)
bkg_mean_336W = aperstats_336W.mean

aperture_area_336W = apertures_336W.area_overlap(section1_336W)
total_bkg_336W = bkg_mean_336W * aperture_area_336W

# perform aperture photometry
star_data_336W = aperture_photometry(section1_336W, apertures_336W)
star_data_336W['x'] = positions_336W[:, 0]
star_data_336W['y'] = positions_336W[:, 1]
star_data_336W['total_bkg'] = total_bkg_336W
star_data_336W['net_flux'] = star_data_336W['aperture_sum'] - star_data_336W['total_bkg']

In [ ]:
zeropoint_336W = 24.692

good_flux_336W = (star_data_336W['net_flux'] > 0) & np.isfinite(star_data_336W['net_flux'])
print(f"Stars removed due to negative flux: {(~good_flux_336W).sum()}")
print(f"Stars remaining: {good_flux_336W.sum()}")
star_data_336W = star_data_336W[good_flux_336W]

star_data_336W['magnitude'] = zeropoint_336W - 2.5 * np.log10(star_data_336W['net_flux'] / t_stack_336W)

for col in star_data_336W.colnames:
    star_data_336W[col].info.format = '%.8g'
star_data_336W[:10].pprint(max_width=-1)

## Matching Stars Between the Two Filters

The F275W and F336W images were taken separately, so the same star will appear
at slightly different pixel positions in each image. I use a **KD Tree** to find the nearest neighbour match for each
star between the two catalogues.

Only accept a match if the two detections are within 3 pixels of each other.
Anything further apart is likely a different star or a bad detection.
We also check the match works in both directions, so that two F336W stars are never matched to the same F275W star.

In [ ]:
pos_275 = np.transpose((np.array(star_data_275W['x']), np.array(star_data_275W['y'])))
pos_336 = np.transpose((np.array(star_data_336W['x']), np.array(star_data_336W['y'])))

# KD Trees
tree_275 = cKDTree(pos_275)
tree_336 = cKDTree(pos_336)

# nearest 275W star for every 336W star, and nearest 336W star for every 275W star
distances, idx = tree_275.query(pos_336, k=1)
_, idx_back = tree_336.query(pos_275, k=1)

# keep: within 3 pixels AND the 275W star also points back to the same 336W star
matched = (distances < 3.0) & (idx_back[idx] == np.arange(len(pos_336)))

print(f"Stars in F275W:       {len(star_data_275W)}")
print(f"Stars in F336W:       {len(star_data_336W)}")
print(f"Matched stars:        {matched.sum()}")

## Plotting the Colour-Magnitude Diagram

The colour is computed by subtracting `F336W magnitude` from `F275W magnitude` and
plotted against the F275W magnitude.

The y-axis is inverted because in astronomy, lower magnitude numbers mean brighter stars —
so brighter stars sit at the top. The x-axis shows colour, where smaller values mean
bluer (hotter) stars.

In [ ]:
# matched is a mask over 336W stars
mag_336 = np.array(star_data_336W['magnitude'])[matched]

# idx[matched] gives the index into 275W for each matched 336W star
mag_275 = np.array(star_data_275W['magnitude'])[idx[matched]]

colour = mag_275 - mag_336

# plot CMD
plt.figure(figsize=(6, 7))
plt.scatter(colour, mag_275, s=2, color='black', alpha=0.5)
plt.gca().invert_yaxis()
plt.xlabel('Colour (F275W - F336W)', fontsize=12)
plt.ylabel('Magnitude F275W', fontsize=12)
plt.title('NGC1261 Colour Magnitude Diagram', fontsize=14)
plt.savefig('cmd_ngc1261.png', dpi=200, bbox_inches='tight')
plt.show()